# 09 · Inspect the project database

Run the first Airflow DAG before querying the tables.

## Imports

In [7]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text

sys.path.insert(0, str(Path('../src').resolve()))
from property_ml.settings import engine

db = engine()


## Batches

In [8]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT * FROM raw.batches ORDER BY group_number, batch_number'), conn))


,group_number,batch_number,run_id,received_at,row_count,status,received_columns,valid_rows,rejected_rows,duplicate_rows,quality
0,4,0,manual__2026-11-27T02:14:41.342275+00:00,2026-11-27 02:14:33.895549+00:00,73784,cleaned,"[acre_lot, bath, bed, brokered_by, city, house...",73784,0,0,"{'missing': {'bed': 0, 'bath': 0, 'city': 0, '..."
1,4,1,manual__2026-11-27T02:17:33.907363+00:00,2026-11-27 02:17:05.248090+00:00,94551,cleaned,"[acre_lot, bath, bed, brokered_by, city, house...",94551,0,0,"{'missing': {'bed': 0, 'bath': 0, 'city': 0, '..."


## Raw and clean counts

In [9]:
with db.connect() as conn:
    for table in ['raw.properties', 'clean.properties', 'audit.inference_events']:
        print(table, conn.execute(text(f'SELECT count(*) FROM {table}')).scalar())


raw.properties 168335
clean.properties 168335
audit.inference_events 94


## Decisions and model history

In [10]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT * FROM audit.batch_decisions ORDER BY batch_number'), conn))
    display(pd.read_sql_query(text('SELECT * FROM audit.training_events ORDER BY id'), conn))


,group_number,batch_number,created_at,valid_rows,pending_rows,decision,reason,model_version,candidate_mae,champion_mae,promotion,run_id,metrics,report_key
0,4,0,2026-11-27 02:15:15.956691+00:00,73784,73784,train,initial baseline,1,44278.238084,NaN,initial model,manual__2026-11-27T02:14:41.342275+00:00,"{'psi': {}, 'schema': {'columns': ['acre_lot',...",group-4/batch-0/report.json
1,4,1,2026-11-27 02:17:55.206305+00:00,94551,94551,train,"bed distribution, bath distribution, house_siz...",2,44933.593909,47261.429877,lower validation MAE,manual__2026-11-27T02:17:33.907363+00:00,"{'psi': {'bed': 1.1983245484975218, 'bath': 1....",group-4/batch-1/report.json


,id,group_number,batch_number,created_at,mlflow_run_id,model_version,candidate_mae,champion_mae,promoted,reason,train_rows
0,1,4,0,2026-11-27 02:16:03.707335+00:00,45b754ad5d214dc2b482d2e2be98307f,1,44278.238084,NaN,True,initial model,44123
1,2,4,1,2026-11-27 02:18:37.216634+00:00,6dcdf3f4f7944fe598a0ad76bfe02d0e,2,44933.593909,47261.429877,True,lower validation MAE,100966


## Sample predictions

In [11]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT created_at, prediction, model_version, pod, status, error FROM audit.inference_events ORDER BY id DESC LIMIT 10'), conn))


,created_at,prediction,model_version,pod,status,error
0,2026-11-27 02:22:59.823506+00:00,241387.765625,2,property-api-bb86f9f5b-vvw7x,success,None
1,2026-11-27 02:21:51.896754+00:00,241387.765625,2,property-api-bb86f9f5b-vvw7x,success,None
2,2026-11-27 02:21:51.889941+00:00,241387.765625,2,property-api-bb86f9f5b-vvw7x,success,None
3,2026-11-27 02:21:51.525451+00:00,241387.765625,2,property-api-bb86f9f5b-vvw7x,success,None
4,2026-11-27 02:21:50.480121+00:00,241387.765625,2,property-api-bb86f9f5b-vvw7x,success,None
5,2026-11-27 02:21:50.224107+00:00,241387.765625,2,property-api-bb86f9f5b-vvw7x,success,None
6,2026-11-27 02:21:50.152327+00:00,241387.765625,2,property-api-bb86f9f5b-stbft,success,None
7,2026-11-27 02:21:50.100360+00:00,241387.765625,2,property-api-bb86f9f5b-stbft,success,None
8,2026-11-27 02:21:49.838965+00:00,241387.765625,2,property-api-bb86f9f5b-stbft,success,None
9,2026-11-27 02:21:49.476824+00:00,241387.765625,2,property-api-bb86f9f5b-vvw7x,success,None


## Training data coverage

In [12]:
with db.connect() as conn:
    display(pd.read_sql_query(text('''
        WITH available AS (
            SELECT group_number, count(*) AS clean_rows
            FROM clean.properties GROUP BY group_number
        ), champions AS (
            SELECT DISTINCT ON (group_number) group_number, batch_number, model_version, (to_jsonb(t)->>'train_rows')::int AS train_rows
            FROM audit.training_events t WHERE promoted
            ORDER BY group_number, batch_number DESC
        )
        SELECT a.group_number, a.clean_rows, c.batch_number, c.model_version, c.train_rows,
               round(100.0 * c.train_rows / nullif(a.clean_rows, 0), 1) AS train_pct_of_current_clean
        FROM available a LEFT JOIN champions c USING (group_number)
    '''), conn))
db.dispose()


,group_number,clean_rows,batch_number,model_version,train_rows,train_pct_of_current_clean
0,4,168335,1,2,100966,60.0
